# 3. EVT-агент: исследование хвоста

**Цели:** поручить агенту библиотеки Graph-EVT-agent воспроизводимый POT-анализ. LLM планирует и объясняет исследование, а численные результаты должны возвращаться как структурированные артефакты.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/fractal_extreme_series.csv").exists())
DATA = ROOT / "data/fractal_extreme_series.csv"
SEED = 42
rng = np.random.default_rng(SEED)
df = pd.read_csv(DATA)
x = df["value"].to_numpy()
assert np.isfinite(x).all() and x.ndim == 1
# Graph-EVT-agent is the analysis library; no model code is copied into this repository.
from graph_evt_agent import ResearchAgent, ResearchTask, LLMConfig

llm = LLMConfig.from_env()  # provider/model/key are selected with environment variables
agent = ResearchAgent(llm=llm, seed=SEED)


In [ ]:
task = ResearchTask(
    question="Какие положительные и отрицательные экстремальные режимы есть в одномерном ряду?",
    data=x,
    methods=["threshold_stability", "mean_excess", "gpd", "declustering"],
    constraints={"min_exceedances": 40, "seed": SEED, "two_sided": True},
)
result = agent.run(task)
result.tables["tail_fits"]


In [ ]:
# Сохраняем машинно-проверяемый след отдельно от текста LLM.
artifact_dir=ROOT/'results'; artifact_dir.mkdir(exist_ok=True)
result.save(artifact_dir/'03_tail_research')
result.summary


Проверяйте число независимых кластеров, устойчивость параметра формы и неопределённость. Ответ LLM не заменяет диагностические таблицы.

**Самопроверка:** зачем declustering? Каков bias–variance порога? Почему анализируются оба хвоста? Где хранится provenance результата?